In [1]:
import pandas as pd
import numpy as np

np.random.seed(12345)

# SET UP POLICIES TABLE
n_policies = 10000

states = ["CA", "TX", "FL", "NY", "PA", "IL", "WA", "AZ", "OH", "NC"]
vehicle_types = ["Sedan", "SUV", "Truck", "Sports", "Hybrid"]

state = np.random.choice(states, n_policies)
driver_age = np.random.randint(18, 75, n_policies)
vehicle = np.random.choice(vehicle_types, n_policies)

policies = pd.DataFrame({
    "policy_id": range(1, n_policies + 1),
    "state": state,
    "driver_age": driver_age,
    "vehicle_type": vehicle,
})

# CALCULATE CLAIM PROBABILITY AND PREMIUM BASED ON FACTORS (expectations)
base_frequency = 0.07
base_severity = 4000

age_factor = np.where(driver_age < 25, 1.6, np.where(driver_age > 65, 1.6, 1)) #suppose younger and older drivers are more likely to have a claim

vehicle_factor = np.where(vehicle == "Sports", 1.8, np.where(vehicle == "SUV", 1.2, 1)) #suppose sports cars and suv's are more likely to have a claim

policies["claim_probability"] = base_frequency * age_factor * vehicle_factor

expected_severity = base_severity * np.where(vehicle == "Sports", 2.5, 1) #suppose fancy sports cars have expensive repairs

policies["annual_premium"] = policies["claim_probability"] * expected_severity # FUNDAMENTAL IDEA: premium = frequency * severity

# SET UP CLAIMS TABLE
has_claim = np.random.binomial(1, policies["claim_probability"]).astype(bool)
n_claims = sum(has_claim)
claims = policies[has_claim].copy()
claims["claim_id"] = range(1, n_claims + 1)
claims = claims[["claim_id", "policy_id", "vehicle_type", "state"]] #vehicle_type and state are temporary and will be removed from the table later

# CALCULATE CLAIM AMOUNTS AND SEVERITY TYPE BASED ON FACTORS (real data)
vehicle_sev_factor = np.where(claims["vehicle_type"] == "Sports", 2, 1) #fancy sports cars have expensive repairs (but less than expected)

claims["claim_amount"] = np.random.lognormal(mean=8, sigma=0.7, size=n_claims) * vehicle_sev_factor #lognormal is skewed right
                                                                                    #(high number of small claims, low number of huge claims)
theft_states = ["CA", "TX", "FL", "NY"]
rand_vals = np.random.rand(n_claims)
claims["peril_type"] = np.where(
    claims["state"].isin(theft_states),
    np.where(rand_vals < 0.4, "Theft", "Collision"),
    np.where(rand_vals < 0.15, "Theft", "Collision")
) #some states have more theft

claims["claim_amount"] = np.round(np.where(claims["peril_type"] == "Theft", claims["claim_amount"]/3, claims["claim_amount"]), 2) #theft claims are
                                                                                                                                #generally cheaper
# the following are helper columns to generate claim_date and will be removed later
months = np.arange(1, 13)
weights = np.array([1.4, 1.2, 1, 0.9, 0.8, 0.8, 0.9, 1, 0.9, 1, 1.2, 1.5])
weights = weights / weights.sum()
claims["claim_month"] = np.random.choice(months, size = len(claims), p = weights) #more claims during travel and winter months
claims["claim_day"] = np.random.randint(1, 29, len(claims)) #only using 28 days to avoid problems with Feb

claims["claim_date"] = pd.to_datetime({
    "year":2025,
    "month":claims["claim_month"],
    "day":claims["claim_day"]}) #generating claim_date column

claims = claims[["claim_id", "claim_date", "policy_id", "claim_amount", "peril_type"]] #removing temporary columns

print(policies)
print(claims)

      policy_id state  driver_age vehicle_type  claim_probability  \
0             1    FL          20          SUV             0.1344   
1             2    IL          50        Sedan             0.0700   
2             3    TX          18       Sports             0.2016   
3             4    PA          56       Sports             0.1260   
4             5    NC          50       Hybrid             0.0700   
...         ...   ...         ...          ...                ...   
9995       9996    NC          49        Truck             0.0700   
9996       9997    WA          60       Sports             0.1260   
9997       9998    IL          70        Sedan             0.1120   
9998       9999    WA          35          SUV             0.0840   
9999      10000    AZ          25          SUV             0.0840   

      annual_premium  
0              537.6  
1              280.0  
2             2016.0  
3             1260.0  
4              280.0  
...              ...  
9995      

In [2]:
print("Observed frequency:", n_claims / n_policies)
print("Average severity:", claims["claim_amount"].mean())
print("Loss ratio:", claims["claim_amount"].sum() / policies["annual_premium"].sum())
print("Average premium:", policies["annual_premium"].mean())

Observed frequency: 0.0949
Average severity: 4075.030073761854
Loss ratio: 0.6812989013677689
Average premium: 567.62216


In [3]:
policies.to_csv("policies.csv", index=False)
claims.to_csv("claims.csv", index=False)